# Train YOLO11 Segmentation on the Normal Rice Dataset

This notebook trains an **Ultralytics YOLO11 instance segmentation** model on `datasets/normal_image_datasets`.

- Task: **instance segmentation**
- Default model: `yolo11n-seg.pt`
- Why start with nano: it is the safest baseline for **Google Colab free tier**, small datasets, and quick thesis-demo iterations.

After you get one stable baseline run, you can switch to `yolo11s-seg.pt` with a single-line config change.

## 1. Mount Google Drive

Run this cell first. If Colab asks you to sign in, finish that before moving on.


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive mount command finished.')
except ModuleNotFoundError:
    print('Not running inside Google Colab. Skipping Google Drive mount.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mount command finished.


## 2. Confirm Colab Runtime

This cell shows whether the notebook can see Colab and Google Drive paths.


In [3]:
import sys
from pathlib import Path

RUNNING_IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Google Colab: {RUNNING_IN_COLAB}')
print(f'Current working directory: {Path.cwd()}')
print(f"/content exists: {Path('/content').exists()}")
print(f"/content/drive exists: {Path('/content/drive').exists()}")
print(f"/content/drive/MyDrive exists: {Path('/content/drive/MyDrive').exists()}")


Running in Google Colab: True
Current working directory: /content
/content exists: True
/content/drive exists: True
/content/drive/MyDrive exists: True


## 3. Find the Project Folder

This cell finds the repo root and confirms the expected dataset folder exists before installation starts.


In [ ]:
import sys
from pathlib import Path

DATASET_NAME = 'normal_image_datasets'
PROJECT_HINT = 'ai-training'


def find_project_root(dataset_name: str, project_hint: str = PROJECT_HINT) -> Path | None:
    preferred_roots = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path('/content') / project_hint,
        Path('/content/drive/MyDrive') / project_hint,
    ]

    checked = set()
    for root in preferred_roots:
        if root in checked or not root.exists():
            continue
        checked.add(root)
        if (root / 'training_utils.py').exists() and (root / 'datasets' / dataset_name).exists():
            return root

    search_roots = [Path('/content'), Path('/content/drive/MyDrive')]
    for base in search_roots:
        if not base.exists() or not base.is_dir():
            continue
        try:
            for candidate in base.rglob('training_utils.py'):
                root = candidate.parent
                if (root / 'datasets' / dataset_name).exists():
                    return root
        except OSError:
            continue

    return None


PROJECT_ROOT = find_project_root(DATASET_NAME)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find a project folder containing training_utils.py and datasets/{DATASET_NAME}."
    )

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f'Resolved project root: {PROJECT_ROOT}')
print(f"training_utils.py exists: {(PROJECT_ROOT / 'training_utils.py').exists()}")
print(f"Dataset exists: {(PROJECT_ROOT / 'datasets' / DATASET_NAME).exists()}")
print(f"Dataset YAML exists: {(PROJECT_ROOT / 'datasets' / DATASET_NAME / 'data.yaml').exists()}")


## 4. Install Requirements and Bootstrap

Only run this after the previous cells look correct. This is the step that installs dependencies and collects runtime info.


In [ ]:
from pathlib import Path

from training_utils import bootstrap_notebook

BOOTSTRAP = bootstrap_notebook(
    dataset_name='normal_image_datasets',
    project_hint='ai-training',
    auto_mount_drive=False,
    quiet_pip=True,
)
PROJECT_ROOT = Path(BOOTSTRAP['project_root'])

print(f"Python version: {BOOTSTRAP['python_version']}")
print(f"Working directory: {BOOTSTRAP['working_directory']}")
print(f"Resolved project root: {PROJECT_ROOT}")
print(f"Running in Google Colab: {BOOTSTRAP['running_in_colab']}")
print(f"Mounted Google Drive in this session: {BOOTSTRAP['mounted_drive']}")
print(f"CUDA available: {BOOTSTRAP['cuda_available']}")
if BOOTSTRAP['gpu_name']:
    print(f"GPU: {BOOTSTRAP['gpu_name']}")
else:
    print('GPU not detected. Training will run on CPU unless you connect a GPU runtime.')


## 5. Imports and Shared Helpers


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import math
import os
import random
import shutil
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from ultralytics import YOLO

from training_utils import (
    ensure_dataset_yaml,
    extract_prediction_table,
    find_best_weights_path,
    is_colab,
    plot_sample_annotations,
    save_metrics_summary,
    seed_everything,
    sigmoid,
    validate_yolo_segmentation_dataset,
)

plt.style.use('seaborn-v0_8-whitegrid')

## 3. Configuration

The defaults below are intentionally conservative for a small dataset and Colab free tier.

If you get a CUDA out-of-memory error, reduce `BATCH` from `16` to `8` or `4`.

In [ ]:
DATASET_DIR = PROJECT_ROOT / 'datasets' / 'normal_image_datasets'
DATA_YAML = DATASET_DIR / 'data.yaml'
EXPECTED_CLASS_NAMES = [
    'broken',
    'chalky',
    'whole',
    'damaged',
    'discolored',
    'foreign',
    'paddy',
    'red',
]

MODEL_WEIGHTS = 'yolo11n-seg.pt'  # Change to 'yolo11s-seg.pt' only after a successful nano baseline.
IMAGE_SIZE = 640
EPOCHS = 100
BATCH = 16
PATIENCE = 20
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = 2
RUN_NAME = 'normal_yolo11n_seg_640'
PROJECT_DIR = PROJECT_ROOT / 'runs_segment'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / RUN_NAME
SAMPLE_SPLIT = 'train'
SAMPLE_COUNT = 4
INFERENCE_LIMIT = 5
CONF = 0.25
IOU = 0.50
SEED = 42
MANUAL_IMAGE_PATH = None  # Example: PROJECT_ROOT / 'some_image.jpg'

seed_everything(SEED)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset directory: {DATASET_DIR}')
print(f'Data YAML: {DATA_YAML}')
print(f'Artifacts directory: {ARTIFACTS_DIR}')
print(f'Running in Colab according to helper: {is_colab()}')

## 4. Dataset Validation

This step checks the folder structure, split files, empty labels, and YAML class definitions before training.

If the dataset YAML already exists, this notebook validates it first and **does not silently override it**.

In [ ]:
yaml_created = False
DATA_YAML = ensure_dataset_yaml(DATASET_DIR, DATA_YAML, EXPECTED_CLASS_NAMES)
    yaml_created = True
    print('Generated data.yaml because it was missing:')
    print(DATA_YAML.read_text(encoding='utf-8'))

validation = validate_yolo_segmentation_dataset(DATASET_DIR, DATA_YAML)
yaml_class_names = validation['class_names']
normalized_expected = [name.strip() for name in EXPECTED_CLASS_NAMES]
normalized_yaml = [str(name).strip() for name in yaml_class_names]

if yaml_class_names:
    class_names = list(yaml_class_names)
    if normalized_yaml != normalized_expected:
        print('WARNING: Dataset YAML class names do not match the recommended normal-dataset class list.')
        print(f'Expected: {EXPECTED_CLASS_NAMES}')
        print(f'YAML:      {yaml_class_names}')
else:
    class_names = EXPECTED_CLASS_NAMES
    print('Dataset YAML has no class names. Falling back to the recommended class list.')

print(f'Using class names for training: {class_names}')
display(validation['summary_df'])

if validation['warnings']:
    print('\nValidation warnings:')
    for warning in validation['warnings']:
        print(f'- {warning}')
else:
    print('No structural dataset issues were detected.')


## 5. Sample Visualization

Always inspect a few samples before training. Annotation mistakes are expensive and easy to miss otherwise.

In [ ]:
selected_samples = plot_sample_annotations(DATASET_DIR, split=SAMPLE_SPLIT, num_samples=SAMPLE_COUNT)
sample_preview_path = ARTIFACTS_DIR / 'sample_annotations.png'
plt.savefig(sample_preview_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved sample preview to: {sample_preview_path}')
print('Selected sample files:')
for path in selected_samples:
    print(f'- {path.name}')

In [ ]:
from pathlib import Path
import yaml

dataset_dir = Path("/content/drive/MyDrive/ai-training/datasets/normal_image_datasets")
yaml_path = dataset_dir / "data.yaml"

data = yaml.safe_load(yaml_path.read_text())
data["path"] = str(dataset_dir.resolve())
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print(yaml_path.read_text())


## 6. Training

This uses transfer learning from `yolo11n-seg.pt` and keeps the defaults conservative for a small segmentation dataset.

In [ ]:
model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    pretrained=True,
    verbose=True,
    seed=SEED,
    deterministic=True,
    exist_ok=True,
)
train_results

## 7. Validation and Test Evaluation

In [ ]:
def collect_metric_snapshot(metrics_obj, prefix):
    snapshot = {}
    results_dict = getattr(metrics_obj, 'results_dict', {}) or {}
    for key, value in results_dict.items():
        if isinstance(value, (int, float, np.floating)):
            snapshot[f'{prefix}_{key}'] = float(value)
        else:
            snapshot[f'{prefix}_{key}'] = value

    for head_name in ('box', 'seg'):
        head = getattr(metrics_obj, head_name, None)
        if head is None:
            continue
        for metric_name in ('map', 'map50', 'map75'):
            value = getattr(head, metric_name, None)
            if value is not None:
                snapshot[f'{prefix}_{head_name}_{metric_name}'] = float(value)

    speed = getattr(metrics_obj, 'speed', None)
    if isinstance(speed, dict):
        for key, value in speed.items():
            snapshot[f'{prefix}_speed_{key}'] = float(value)
    return snapshot


best_model_path = find_best_weights_path(PROJECT_DIR, RUN_NAME)
best_model = YOLO(str(best_model_path))

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(PROJECT_DIR),
    name=f'{RUN_NAME}_val',
    exist_ok=True,
)

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(PROJECT_DIR),
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)

metrics_summary = {
    'run_name': RUN_NAME,
    'model_weights': MODEL_WEIGHTS,
    'best_model_path': str(best_model_path),
    'dataset_dir': str(DATASET_DIR),
    'data_yaml': str(DATA_YAML),
    'image_size': IMAGE_SIZE,
    'epochs': EPOCHS,
    'batch': BATCH,
    'patience': PATIENCE,
}
metrics_summary.update(collect_metric_snapshot(val_metrics, 'val'))
metrics_summary.update(collect_metric_snapshot(test_metrics, 'test'))

metrics_json_path = save_metrics_summary(ARTIFACTS_DIR / 'metrics_summary', metrics_summary)
display(pd.DataFrame([metrics_summary]).T.rename(columns={0: 'value'}))
print(f'Saved metrics summary to: {metrics_json_path}')

## 8. Inference Demo

This section runs the trained model on a few test images plus one optional manual image path if you provide it.

In [ ]:
test_images = sorted((DATASET_DIR / 'test' / 'images').glob('*'))
test_images = [path for path in test_images if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}]

inference_sources = test_images[:INFERENCE_LIMIT]
if MANUAL_IMAGE_PATH is not None:
    manual_path = Path(MANUAL_IMAGE_PATH)
    if manual_path.exists():
        inference_sources.append(manual_path)
    else:
        print(f'Manual image path does not exist: {manual_path}')

if not inference_sources:
    raise FileNotFoundError('No inference images were found in the test split, and no manual image path was provided.')

prediction_results = best_model.predict(
    source=[str(path) for path in inference_sources],
    conf=CONF,
    iou=IOU,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    save=False,
    verbose=False,
)

preview_dir = ARTIFACTS_DIR / 'inference_previews'
preview_dir.mkdir(parents=True, exist_ok=True)

for result in prediction_results:
    rendered = result.plot()
    output_path = preview_dir / f"{Path(result.path).stem}_pred.jpg"
    cv2.imwrite(str(output_path), rendered)

for result in prediction_results[: min(3, len(prediction_results))]:
    rendered = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(7, 7))
    plt.imshow(rendered)
    plt.title(Path(result.path).name)
    plt.axis('off')
    plt.show()

print(f'Saved preview images to: {preview_dir}')

## 9. Confidence Scoring

Ultralytics already provides a **native confidence score per predicted instance**. That native confidence is what you should use for per-grain predictions.

The `sigmoid()` helper below is only for **optional downstream calibration or custom scoring**, such as turning a hand-designed feature into a bounded 0 to 1 score. It should **not** replace YOLO's built-in confidence output.

In [ ]:
prediction_df = extract_prediction_table(prediction_results, [str(name).strip() for name in class_names])
prediction_csv_path = ARTIFACTS_DIR / 'prediction_table.csv'
prediction_df.to_csv(prediction_csv_path, index=False)

display(prediction_df.head(20))
print(f'Saved per-instance prediction table to: {prediction_csv_path}')

if not prediction_df.empty:
    image_level_df = (
        prediction_df.groupby('image_name', as_index=False)
        .agg(
            grain_count=('image_name', 'size'),
            mean_confidence=('confidence', 'mean'),
            mean_confidence_percent=('confidence_percent', 'mean'),
        )
    )

    count_center = image_level_df['grain_count'].median()
    count_scale = image_level_df['grain_count'].std(ddof=0)
    count_scale = float(count_scale) if count_scale and not np.isnan(count_scale) else 1.0
    image_level_df['normalized_count_deviation'] = (image_level_df['grain_count'] - count_center) / count_scale
    image_level_df['custom_quality_score'] = sigmoid(
        image_level_df['mean_confidence'].fillna(0.0) * 4.0 - image_level_df['normalized_count_deviation'].abs()
    )

    image_level_csv_path = ARTIFACTS_DIR / 'image_level_scores.csv'
    image_level_df.to_csv(image_level_csv_path, index=False)
    display(image_level_df)
    print(f'Saved image-level custom scoring table to: {image_level_csv_path}')
else:
    image_level_df = pd.DataFrame()
    print('No predictions were produced for the selected inference images.')

## 10. Ultralytics Training Summary Figures

This cell displays the standard Ultralytics training summary images directly from the run folder so you can review or screenshot them for the paper.


In [ ]:
run_dir = PROJECT_DIR / RUN_NAME
ultralytics_figure_candidates = [
    ('results.png', 'Ultralytics Training Summary'),
    ('labels.jpg', 'Ultralytics Label Overview'),
    ('confusion_matrix_normalized.png', 'Normalized Confusion Matrix'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png', 'Precision-Recall Curve'),
    ('F1_curve.png', 'F1 Curve'),
    ('P_curve.png', 'Precision Curve'),
    ('R_curve.png', 'Recall Curve'),
]

available_ultralytics_figures = [
    (run_dir / filename, title)
    for filename, title in ultralytics_figure_candidates
    if (run_dir / filename).exists()
]

if not available_ultralytics_figures:
    print(f'No Ultralytics summary figures were found yet in: {run_dir}')
else:
    cols = 2
    rows = math.ceil(len(available_ultralytics_figures) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, (image_path, title) in zip(axes, available_ultralytics_figures):
        axis.imshow(plt.imread(image_path))
        axis.set_title(title)
        axis.axis('off')

    for axis in axes[len(available_ultralytics_figures):]:
        axis.axis('off')

    plt.tight_layout()
    plt.show()

    print('Available Ultralytics summary files:')
    for image_path, title in available_ultralytics_figures:
        print(f'- {title}: {image_path}')


## 11. Paper Figures and Visual Summaries

This section gathers publication-friendly visuals from the training run, dataset structure, and qualitative predictions.


In [ ]:
paper_figures_dir = ARTIFACTS_DIR / 'paper_figures'
paper_figures_dir.mkdir(parents=True, exist_ok=True)

results_csv_path = PROJECT_DIR / RUN_NAME / 'results.csv'
run_dir = PROJECT_DIR / RUN_NAME


def count_labels_by_class(dataset_dir: Path, split: str, class_names: list[str]) -> pd.DataFrame:
    labels_dir = dataset_dir / split / 'labels'
    counts = {class_name: 0 for class_name in class_names}

    if not labels_dir.exists():
        return pd.DataFrame({'class_name': list(counts.keys()), 'count': list(counts.values())})

    for label_file in sorted(labels_dir.glob('*.txt')):
        lines = label_file.read_text(encoding='utf-8').strip().splitlines()
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                class_id = int(float(parts[0]))
            except ValueError:
                continue
            if 0 <= class_id < len(class_names):
                counts[class_names[class_id]] += 1

    return pd.DataFrame({'class_name': list(counts.keys()), 'count': list(counts.values())})


paper_assets = {}

if results_csv_path.exists():
    results_df = pd.read_csv(results_csv_path)
    numeric_df = results_df.apply(pd.to_numeric, errors='ignore')

    loss_columns = [
        column for column in numeric_df.columns
        if any(token in column.lower() for token in ['train/box_loss', 'train/seg_loss', 'train/cls_loss', 'val/box_loss', 'val/seg_loss', 'val/cls_loss'])
    ]
    metric_columns = [
        column for column in numeric_df.columns
        if any(token in column.lower() for token in ['metrics/precision', 'metrics/recall', 'metrics/map50', 'metrics/map50-95'])
    ]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    if loss_columns:
        for column in loss_columns:
            axes[0].plot(numeric_df.index + 1, numeric_df[column], linewidth=2, label=column)
        axes[0].set_title('Training and Validation Loss Curves')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)
    else:
        axes[0].text(0.5, 0.5, 'Loss columns not found in results.csv', ha='center', va='center')
        axes[0].axis('off')

    if metric_columns:
        for column in metric_columns:
            axes[1].plot(numeric_df.index + 1, numeric_df[column], linewidth=2, label=column)
        axes[1].set_title('Precision, Recall, and mAP Curves')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Score')
        axes[1].set_ylim(0, 1.05)
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'Metric columns not found in results.csv', ha='center', va='center')
        axes[1].axis('off')

    plt.tight_layout()
    training_curves_path = paper_figures_dir / 'training_curves_summary.png'
    plt.savefig(training_curves_path, dpi=220, bbox_inches='tight')
    plt.show()
    paper_assets['training_curves_summary'] = str(training_curves_path)
else:
    print(f'results.csv not found at: {results_csv_path}')

split_summary_df = validation['summary_df'].copy()
display(split_summary_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
split_plot_df = split_summary_df.set_index('split')[['images', 'labels']]
split_plot_df.plot(kind='bar', ax=axes[0], rot=0, color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('Dataset Size by Split')
axes[0].set_xlabel('Split')
axes[0].set_ylabel('Count')
axes[0].grid(True, axis='y', alpha=0.3)

class_distribution_frames = []
for split_name in ['train', 'valid', 'test']:
    counts_df = count_labels_by_class(DATASET_DIR, split_name, class_names)
    counts_df['split'] = split_name
    class_distribution_frames.append(counts_df)

class_distribution_df = pd.concat(class_distribution_frames, ignore_index=True)
display(class_distribution_df)
class_pivot = class_distribution_df.pivot(index='class_name', columns='split', values='count').fillna(0)
class_pivot.plot(kind='bar', ax=axes[1], rot=30)
axes[1].set_title('Class Instance Distribution by Split')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Annotated instances')
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
dataset_summary_path = paper_figures_dir / 'dataset_summary.png'
plt.savefig(dataset_summary_path, dpi=220, bbox_inches='tight')
plt.show()
paper_assets['dataset_summary'] = str(dataset_summary_path)

figure_candidates = [
    ('labels.jpg', 'Ultralytics Label Overview'),
    ('results.png', 'Ultralytics Training Summary'),
    ('confusion_matrix_normalized.png', 'Normalized Confusion Matrix'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png', 'Precision-Recall Curve'),
    ('F1_curve.png', 'F1 Curve'),
    ('P_curve.png', 'Precision Curve'),
    ('R_curve.png', 'Recall Curve'),
]

available_figures = [(run_dir / filename, title) for filename, title in figure_candidates if (run_dir / filename).exists()]
if available_figures:
    cols = 2
    rows = math.ceil(len(available_figures) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, (image_path, title) in zip(axes, available_figures):
        image = plt.imread(image_path)
        axis.imshow(image)
        axis.set_title(title)
        axis.axis('off')
        paper_assets[image_path.stem] = str(image_path)
    for axis in axes[len(available_figures):]:
        axis.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No Ultralytics figure files were found yet in the run directory.')

qualitative_images = []
if sample_preview_path.exists():
    qualitative_images.append((sample_preview_path, 'Annotated Training Samples'))

for preview_path in sorted(preview_dir.glob('*_pred.jpg'))[:4]:
    qualitative_images.append((preview_path, f'Prediction: {preview_path.stem}'))

if qualitative_images:
    cols = 2
    rows = math.ceil(len(qualitative_images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, (image_path, title) in zip(axes, qualitative_images):
        image = plt.imread(image_path)
        axis.imshow(image)
        axis.set_title(title)
        axis.axis('off')
    for axis in axes[len(qualitative_images):]:
        axis.axis('off')
    plt.tight_layout()
    qualitative_panel_path = paper_figures_dir / 'qualitative_examples.png'
    plt.savefig(qualitative_panel_path, dpi=220, bbox_inches='tight')
    plt.show()
    paper_assets['qualitative_examples'] = str(qualitative_panel_path)
else:
    print('No qualitative images were available to assemble into a panel.')

paper_assets_path = paper_figures_dir / 'paper_figures_manifest.json'
paper_assets_path.write_text(json.dumps(paper_assets, indent=2), encoding='utf-8')
print(f'Saved paper figure manifest to: {paper_assets_path}')
print(json.dumps(paper_assets, indent=2))


## 12. Export the Best Model

The `.pt` file is the main training artifact. Exporting to ONNX is helpful later for backend inference or deployment experiments.


In [1]:
export_dir = ARTIFACTS_DIR / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)

best_pt_copy = export_dir / best_model_path.name
if best_model_path.resolve() != best_pt_copy.resolve():
    shutil.copy2(best_model_path, best_pt_copy)

exported_onnx_path = None
try:
    raw_export_path = Path(best_model.export(format='onnx', imgsz=IMAGE_SIZE))
    exported_onnx_path = export_dir / raw_export_path.name
    if raw_export_path.exists() and raw_export_path.resolve() != exported_onnx_path.resolve():
        shutil.copy2(raw_export_path, exported_onnx_path)
    elif raw_export_path.exists():
        exported_onnx_path = raw_export_path
    print(f'ONNX export saved to: {exported_onnx_path}')
except Exception as exc:
    print(f'ONNX export skipped because it failed: {exc}')

print(f'Copied best PyTorch weights to: {best_pt_copy}')

NameError: name 'ARTIFACTS_DIR' is not defined

## 13. Save Artifact Manifest


In [ ]:
artifact_manifest = {
    'run_name': RUN_NAME,
    'artifacts_dir': str(ARTIFACTS_DIR),
    'best_model_path': str(best_model_path),
    'metrics_json': str((ARTIFACTS_DIR / 'metrics_summary.json')),
    'metrics_csv': str((ARTIFACTS_DIR / 'metrics_summary.csv')),
    'prediction_csv': str(prediction_csv_path),
    'sample_preview': str(sample_preview_path),
    'inference_preview_dir': str(preview_dir),
    'paper_figures_dir': str(paper_figures_dir),
    'paper_figures_manifest': str(paper_assets_path),
    'export_dir': str(export_dir),
}

manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding='utf-8')
print(f'Saved artifact manifest to: {manifest_path}')
print(json.dumps(artifact_manifest, indent=2))


## 14. Next Steps

- Add more labeled data, especially for visually similar or rare classes.
- Review validation and test predictions for confusion between similar grain defects.
- Keep the nano model as the first deployment candidate because it is easier to train and cheaper to run.
- After you have one clean baseline, try `yolo11s-seg.pt` and compare mask mAP and inference speed.
- If performance plateaus, inspect annotations carefully before increasing model size.
